In [1]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTTrainer
import torch
import json
from peft import prepare_model_for_kbit_training

g:\AI\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
g:\AI\ai_env\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [3]:
MODEL_NAME = "Qwen/Qwen3-8B"


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [5]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

Loading checkpoint shards: 100%|██████████| 5/5 [01:51<00:00, 22.30s/it]


In [3]:
dataset = load_dataset(
    "json",
    data_files= "v3a_dataset_clean.jsonl"
)

Generating train split: 40763 examples [00:00, 65983.73 examples/s]


DatasetGenerationError: An error occurred while generating the dataset

In [7]:
import json
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

INPUT_FILE = "./dataset/V3A.jsonl"
OUTPUT_FILE = "v3a_dataset_clean.jsonl"
BAD_FILE = "v3a_dataset_bad.jsonl"


# ============================================================
# EXPECTED PROFILE SCHEMA
# ============================================================

PROFILE_FIELDS = {
    "profession": "",
    "career_goal": "",
    "education": "",
    "interests": [],
    "communication_preferences": [],
    "learning_style": "",
    "personality_traits": [],
    "lifestyle": [],
    "languages": [],
    "values": [],
    "social_preferences": [],
    "relationship_preferences": [],
    "work_preferences": [],
    "goals": [],
    "stable_habits": [],
    "preferences": [],
    "important_constraints": [],
    "confidence": 0.0,
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_string(value):
    """Return a clean string or empty string."""
    if value is None:
        return ""

    if isinstance(value, str):
        return value.strip()

    return str(value).strip()


def clean_list(value):
    """Return a clean list of strings."""
    if value is None:
        return []

    if not isinstance(value, list):
        return []

    result = []

    for item in value:
        if item is None:
            continue

        if isinstance(item, str):
            item = item.strip()

            if item:
                result.append(item)

    return result


def clean_confidence(value):
    """Ensure confidence is a float between 0 and 1."""
    try:
        value = float(value)
    except (TypeError, ValueError):
        return 0.0

    return max(0.0, min(1.0, value))


# ============================================================
# CLEAN CONVERSATION UNDERSTANDING
# ============================================================

def clean_conversation_understanding(value):
    """
    Normalize conversation_understanding.

    IMPORTANT:
    current_goal -> intent

    Final schema:

    {
        "topic": "",
        "intent": ""
    }
    """

    if not isinstance(value, dict):
        return {
            "topic": "",
            "intent": ""
        }

    topic = clean_string(value.get("topic", ""))

    # New canonical field
    intent = clean_string(value.get("intent", ""))

    # Old field -> new field
    if not intent:
        intent = clean_string(value.get("current_goal", ""))

    return {
        "topic": topic,
        "intent": intent
    }


# ============================================================
# CLEAN PROFILE
# ============================================================

def clean_profile(profile):
    """Normalize the profile into the exact V3A schema."""

    if not isinstance(profile, dict):
        return None

    cleaned = {}

    string_fields = [
        "profession",
        "career_goal",
        "education",
        "learning_style",
    ]

    list_fields = [
        "interests",
        "communication_preferences",
        "personality_traits",
        "lifestyle",
        "languages",
        "values",
        "social_preferences",
        "relationship_preferences",
        "work_preferences",
        "goals",
        "stable_habits",
        "preferences",
        "important_constraints",
    ]

    for field in string_fields:
        cleaned[field] = clean_string(profile.get(field, ""))

    for field in list_fields:
        cleaned[field] = clean_list(profile.get(field, []))

    cleaned["confidence"] = clean_confidence(
        profile.get("confidence", 0.0)
    )

    return cleaned


# ============================================================
# CLEAN ONE RECORD
# ============================================================

def clean_record(record):
    """Clean and validate one dataset record."""

    if not isinstance(record, dict):
        return None, "Record is not a JSON object"

    # --------------------------------------------------------
    # Required top-level fields
    # --------------------------------------------------------

    if record.get("task") != "v3a_user_profile_intelligence":
        return None, "Invalid task"

    if not isinstance(record.get("input"), dict):
        return None, "Missing or invalid input"

    if not isinstance(record.get("output"), dict):
        return None, "Missing or invalid output"

    # --------------------------------------------------------
    # Input
    # --------------------------------------------------------

    input_data = record["input"]

    conversation = clean_string(
        input_data.get("conversation", "")
    )

    conversation_summary = clean_string(
        input_data.get("conversation_summary", "")
    )

    conversation_understanding = clean_conversation_understanding(
        input_data.get("conversation_understanding", {})
    )

    user_memories = clean_list(
        input_data.get("user_memories", [])
    )

    # Conversation is required
    if not conversation:
        return None, "Empty conversation"

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    profile = clean_profile(
        record["output"].get("profile")
    )

    if profile is None:
        return None, "Invalid profile"

    # --------------------------------------------------------
    # FINAL FIXED STRUCTURE
    # --------------------------------------------------------

    cleaned_record = {
        "task": "v3a_user_profile_intelligence",

        "instruction": (
            "Generate or update the user's long-term profile."
        ),

        "input": {
            "conversation": conversation,

            "conversation_summary": conversation_summary,

            "conversation_understanding": {
                "topic": conversation_understanding["topic"],
                "intent": conversation_understanding["intent"]
            },

            "user_memories": user_memories
        },

        "output": {
            "profile": profile
        }
    }

    return cleaned_record, None


# ============================================================
# PROCESS DATASET
# ============================================================

def clean_dataset():

    input_path = Path(INPUT_FILE)
    output_path = Path(OUTPUT_FILE)
    bad_path = Path(BAD_FILE)

    if not input_path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {input_path}"
        )

    total = 0
    valid = 0
    bad = 0

    bad_records = []

    print("=" * 70)
    print("V3A DATASET CLEANER")
    print("=" * 70)

    with open(
        input_path,
        "r",
        encoding="utf-8"
    ) as infile, open(
        output_path,
        "w",
        encoding="utf-8"
    ) as outfile:

        for line_number, line in enumerate(
            infile,
            start=1
        ):

            total += 1

            line = line.strip()

            if not line:
                continue

            # ------------------------------------------------
            # Parse JSON
            # ------------------------------------------------

            try:
                record = json.loads(line)

            except json.JSONDecodeError as e:

                bad += 1

                bad_records.append({
                    "line": line_number,
                    "error": f"Invalid JSON: {e}",
                    "raw": line
                })

                continue

            # ------------------------------------------------
            # Clean record
            # ------------------------------------------------

            cleaned, error = clean_record(record)

            if cleaned is None:

                bad += 1

                bad_records.append({
                    "line": line_number,
                    "error": error,
                    "raw": record
                })

                continue

            # ------------------------------------------------
            # Write clean record
            # ------------------------------------------------

            outfile.write(
                json.dumps(
                    cleaned,
                    ensure_ascii=False
                ) + "\n"
            )

            valid += 1

    # ========================================================
    # SAVE BAD RECORDS
    # ========================================================

    with open(
        bad_path,
        "w",
        encoding="utf-8"
    ) as f:

        for record in bad_records:

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

    # ========================================================
    # REPORT
    # ========================================================

    print()
    print("=" * 70)
    print("CLEANING COMPLETE")
    print("=" * 70)

    print(f"Total records : {total:,}")
    print(f"Valid records : {valid:,}")
    print(f"Bad records   : {bad:,}")

    print()
    print(f"Clean dataset : {output_path}")
    print(f"Bad records   : {bad_path}")

    print("=" * 70)


if __name__ == "__main__":
    clean_dataset()

V3A DATASET CLEANER

CLEANING COMPLETE
Total records : 67,509
Valid records : 67,504
Bad records   : 5

Clean dataset : v3a_dataset_clean.jsonl
Bad records   : v3a_dataset_bad.jsonl


In [10]:
from datasets import load_dataset, Features, Value, Sequence

features = Features({
    "task": Value("string"),
    "instruction": Value("string"),

    "input": {
        "conversation": Value("string"),
        "conversation_summary": Value("string"),

        "conversation_understanding": {
            "topic": Value("string"),
            "intent": Value("string"),
        },

        "user_memories": Sequence(Value("string")),
    },

    "output": {
        "profile": {
            "profession": Value("string"),
            "career_goal": Value("string"),
            "education": Value("string"),

            "interests": Sequence(Value("string")),
            "communication_preferences": Sequence(Value("string")),

            "learning_style": Value("string"),

            "personality_traits": Sequence(Value("string")),
            "lifestyle": Sequence(Value("string")),
            "languages": Sequence(Value("string")),
            "values": Sequence(Value("string")),
            "social_preferences": Sequence(Value("string")),
            "relationship_preferences": Sequence(Value("string")),
            "work_preferences": Sequence(Value("string")),
            "goals": Sequence(Value("string")),
            "stable_habits": Sequence(Value("string")),
            "preferences": Sequence(Value("string")),
            "important_constraints": Sequence(Value("string")),

            "confidence": Value("float32"),
        }
    }
})


dataset = load_dataset(
    "json",
    data_files="v3a_dataset_clean.jsonl",
    split="train",
    features=features
)

print(dataset)
print(dataset.features)

Generating train split: 67504 examples [00:00, 78994.40 examples/s]

Dataset({
    features: ['task', 'instruction', 'input', 'output'],
    num_rows: 67504
})
{'task': Value('string'), 'instruction': Value('string'), 'input': {'conversation': Value('string'), 'conversation_summary': Value('string'), 'conversation_understanding': {'topic': Value('string'), 'intent': Value('string')}, 'user_memories': List(Value('string'))}, 'output': {'profile': {'profession': Value('string'), 'career_goal': Value('string'), 'education': Value('string'), 'interests': List(Value('string')), 'communication_preferences': List(Value('string')), 'learning_style': Value('string'), 'personality_traits': List(Value('string')), 'lifestyle': List(Value('string')), 'languages': List(Value('string')), 'values': List(Value('string')), 'social_preferences': List(Value('string')), 'relationship_preferences': List(Value('string')), 'work_preferences': List(Value('string')), 'goals': List(Value('string')), 'stable_habits': List(Value('string')), 'preferences': List(Value('string')), 'im

In [15]:
import json

# ==========================================================
# Prepare Model
# ==========================================================

model = prepare_model_for_kbit_training(model)

model.enable_input_require_grads()
model.gradient_checkpointing_enable()
model.config.use_cache = False

# ==========================================================
# System Prompt
# ==========================================================

SYSTEM_PROMPT = """You are an expert assistant objective prediction model.

Your task is to predict the assistant's objective from the given conversation.

Rules:
- Predict only the assistant's objective.
- Do NOT generate a reply.
- Do NOT summarize the conversation.
- Do NOT extract memories.
- Do NOT infer unsupported information.
- Base your prediction only on the provided conversation.

Return ONLY valid JSON in the following format:

{
  "primary_objective": "",
  "secondary_objective": "",
  "priority": "",
  "reason": ""
}

Field Definitions:
- primary_objective: The assistant's main objective.
- secondary_objective: An optional supporting objective. Use "None" if no meaningful secondary objective exists.
- priority: One of "High", "Medium", or "Low".
- reason: A brief explanation (1–2 sentences) describing why these objectives were selected.

Return only the JSON object. Do not include any extra text or markdown.
"""

# ==========================================================
# Formatting Function
# ==========================================================

def formatting_func(example):

    user_input = (
        f"{example['instruction']}\n\n"
        f"{json.dumps(example['input'], ensure_ascii=False, separators=(',', ':'))}"
    )

    assistant_output = json.dumps(
        example["output"],
        ensure_ascii=False,
        separators=(",", ":")
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_input,
        },
        {
            "role": "assistant",
            "content": assistant_output,
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

In [18]:
# Printing  TOKEN ,DISTRIBUTION ,FORMATTING ,LONGEST ,SAMPLES of this dataset (important thing before training)

import time
from statistics import mean, median

def analyze_dataset(name, dataset, tokenizer, formatting_func):
    print("\n" + "="*70)
    print(name)
    print("="*70)

    train = dataset["train"]

    lengths = []
    format_times = []

    start_total = time.time()

    for i, sample in enumerate(train):
        t1 = time.time()

        text = formatting_func(sample)

        t2 = time.time()
        format_times.append(t2 - t1)

        tokens = tokenizer(text, add_special_tokens=True)["input_ids"]
        lengths.append(len(tokens))

        if (i + 1) % 5000 == 0:
            print(f"Processed {i+1}/{len(train)}")

    total_time = time.time() - start_total

    print("\n----- BASIC -----")
    print("Samples              :", len(train))
    print("Average Tokens       :", round(mean(lengths),2))
    print("Median Tokens        :", median(lengths))
    print("Minimum Tokens       :", min(lengths))
    print("Maximum Tokens       :", max(lengths))

    print("\n----- TOKEN DISTRIBUTION -----")
    print(">256 tokens          :", sum(x > 256 for x in lengths))
    print(">512 tokens          :", sum(x > 512 for x in lengths))
    print(">1024 tokens         :", sum(x > 1024 for x in lengths))
    print(">2048 tokens         :", sum(x > 2048 for x in lengths))
    print(">4096 tokens         :", sum(x > 4096 for x in lengths))

    print("\n----- FORMATTING -----")
    print("Formatting Time      :", round(total_time,2), "sec")
    print("Average/sample       :", round(mean(format_times)*1000,3), "ms")
    print("Samples/sec          :", round(len(train)/total_time,2))

    print("\n----- LONGEST SAMPLES -----")
    top = sorted(enumerate(lengths), key=lambda x: x[1], reverse=True)[:10]

    for idx, tok in top:
        print(f"Sample {idx:6d} : {tok} tokens")

    return lengths


In [19]:
old_dataset = load_dataset(
    "json",
    data_files= r"./v5a/V5A.jsonl",
  
)

# new_dataset = load_dataset(
#     "json",
#     data_files=r"./newV4a/finalV4A.jsonl",
  
# )


old_lengths = analyze_dataset(
    "OLD DATASET",
    old_dataset,
    tokenizer,
    formatting_func
)

# new_lengths = analyze_dataset(
#     "NEW V4A DATASET",
#     new_dataset,
#     tokenizer,
#     formatting_func
# )


OLD DATASET
Processed 5000/59921
Processed 10000/59921
Processed 15000/59921
Processed 20000/59921
Processed 25000/59921
Processed 30000/59921
Processed 35000/59921
Processed 40000/59921
Processed 45000/59921
Processed 50000/59921
Processed 55000/59921

----- BASIC -----
Samples              : 59921
Average Tokens       : 446.63
Median Tokens        : 439
Minimum Tokens       : 303
Maximum Tokens       : 879

----- TOKEN DISTRIBUTION -----
>256 tokens          : 59921
>512 tokens          : 10744
>1024 tokens         : 0
>2048 tokens         : 0
>4096 tokens         : 0

----- FORMATTING -----
Formatting Time      : 87.0 sec
Average/sample       : 0.23 ms
Samples/sec          : 688.73

----- LONGEST SAMPLES -----
Sample  49784 : 879 tokens
Sample  24899 : 877 tokens
Sample  54440 : 850 tokens
Sample  26312 : 848 tokens
Sample  17506 : 842 tokens
Sample   9744 : 836 tokens
Sample  17119 : 833 tokens
Sample  39085 : 819 tokens
Sample  48582 : 818 tokens
Sample   4658 : 808 tokens
